# 01 Model Validation & Data Exploration
Loads the pretrained CheXpert DenseNet model and the 250-image NIH ChestX-ray14 subset (50 images per pathology: Cardiomegaly, Effusion, Pneumonia, Pneumothorax, Atelectasis), then validates the full load → preprocess → predict pipeline before any saliency work begins. Checks whether the model classifies these images confidently and correctly, and documents the resulting class-level imbalance for use in later phases.
###
#### Setup

In [1]:
# Import required libraries and project config (paths, seed, device, pathology list)
import torchxrayvision as xrv
import skimage, torch, torchvision
import pandas as pd

In [2]:
# Load the pretrained torchxrayvision DenseNet model, trained on the CheXpert dataset
model = xrv.models.DenseNet(weights="densenet121-res224-chex")

In [3]:
# Inspect the model's output pathology labels, including unused/blank slots for pathologies this checkpoint wasn't trained to predict
print(model.pathologies)
print(model.pathologies.index('Cardiomegaly'))

['Atelectasis', 'Consolidation', '', 'Pneumothorax', 'Edema', '', '', 'Effusion', 'Pneumonia', '', 'Cardiomegaly', '', '', '', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']
10


#### Single-image preprocessing & prediction test

In [4]:
# Test the image loading and preprocessing pipeline (normalisation, center crop, resize to 224×224) on a single example image
img = skimage.io.imread("../data/xray_dataset/00001270_005.png")
img = xrv.datasets.normalize(img, 255) # Convert 8-bit image to [-1024, 1024] range

print(img.shape)
print(img.ndim)

img = img[None, ...] # Equivalent to (1, height, weight) - image is 2 dimensions so need to add in channels

print(img.shape)
print(img.dtype)
print(img.min(), img.max())

(1024, 1024)
2
(1, 1024, 1024)
float32
-1024.0 1024.0


In [5]:
# Run the single preprocessed image through the model and inspect the raw output shape and values as a sanity check
transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
img = transform(img)
img = torch.from_numpy(img)
img = img.unsqueeze(0) # Adds batch dimension to the front: (batch, channels, height, width)

print(img.shape)
print(img.dtype)
print(img.min(), img.max())

output = model(img)
print(output)
print(output.shape)

torch.Size([1, 1, 224, 224])
torch.float32
tensor(-1024.) tensor(1009.2689)
tensor([[0.5585, 0.5065, 0.5000, 0.1179, 0.5184, 0.5000, 0.5000, 0.4514, 0.2241,
         0.5000, 0.2463, 0.5000, 0.5000, 0.5000, 0.2194, 0.5012, 0.5212, 0.3072]],
       grad_fn=<IndexPutBackward0>)
torch.Size([1, 18])


In [6]:
# Output size of sampled dataset and ensure each class has an equivalent number of labels for even class distribution
labels = pd.read_csv('../data/sampled_data_labels.csv')
print(labels.shape)
print(labels.size)
print(labels.value_counts('Finding Labels'))
labels.head(10)

(250, 2)
500
Finding Labels
Cardiomegaly    50
Effusion        50
Pneumonia       50
Pneumothorax    50
Atelectasis     50
Name: count, dtype: int64


,Image Index,Finding Labels
0,00010165_001.png,Cardiomegaly
1,00003395_005.png,Cardiomegaly
2,00002038_001.png,Cardiomegaly
3,00018837_000.png,Cardiomegaly
4,00015770_011.png,Cardiomegaly
5,00018204_000.png,Cardiomegaly
6,00005064_013.png,Cardiomegaly
7,00008806_014.png,Cardiomegaly
8,00030572_000.png,Cardiomegaly
9,00015361_000.png,Cardiomegaly


In [7]:
# Testing the model on a single image example and outputting the probability of predicting the true label
match = labels[labels['Image Index'] == '00001270_005.png']
true_label = match['Finding Labels'].item()
print(true_label)
true_label_index = model.pathologies.index(true_label)
print(true_label_index)
true_label_probability = output[0][true_label_index].data
print(true_label_probability)

Atelectasis
0
tensor(0.5585)


#### Sampled dataset preprocessing & predictions

In [12]:
# Run all 250 images through the model, recording each image's predicted probability for its own true label
probabilties = []
for index, row in labels.iterrows():
    filename = row['Image Index']
    img = skimage.io.imread(f"../data/xray_dataset/{filename}")
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = img[..., :3]  # Remove alpha channel
        img = skimage.color.rgb2gray(img)
    img = xrv.datasets.normalize(img, 255) # Convert 8-bit image to [-1024, 1024] range
    img = img[None, ...] # Equivalent to (1, height, weight) - image is 2 dimensions so need to add in channels

    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    img = transform(img)
    img = torch.from_numpy(img)
    img = img.unsqueeze(0) # Adds batch dimension to the front: (batch, channels, height, width)

    with torch.no_grad():
        output = model(img)

    true_label = row['Finding Labels']
    true_label_index = model.pathologies.index(true_label)

    true_label_probability = output[0][true_label_index].item()
    probabilties.append(true_label_probability)
labels['True Label Probability'] = probabilties
labels.head(20)

labels.to_csv('../data/true_probability_labels.csv', index = False)


#### Summary statistics

In [9]:
# Summarise prediction confidence overall and per pathology, and calculate the proportion of images classified confidently (probability ≥ 0.7)
labels.info()
labels.value_counts(['Finding Labels'])
print('\n')
print(labels['True Label Probability'].describe())

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Image Index             250 non-null    str    
 1   Finding Labels          250 non-null    str    
 2   True Label Probability  250 non-null    float64
dtypes: float64(1), str(2)
memory usage: 6.0 KB


count    250.000000
mean       0.490271
std        0.317858
min        0.004224
25%        0.164518
50%        0.522809
75%        0.765196
max        0.991798
Name: True Label Probability, dtype: float64


In [10]:
threshold = 0.7 
fraction_confident = (labels['True Label Probability'] >= threshold).mean()
print(f"Fraction confidently correct at threshold {threshold}: {fraction_confident:.2%}")
print((labels['True Label Probability'] >= 0.7).groupby(labels['Finding Labels']).sum())

Fraction confidently correct at threshold 0.7: 31.20%
Finding Labels
Atelectasis     23
Cardiomegaly    16
Effusion        37
Pneumonia        2
Pneumothorax     0
Name: True Label Probability, dtype: int64


In [11]:
labels.groupby('Finding Labels')['True Label Probability'].describe()

,count,mean,std,min,25%,50%,75%,max
Finding Labels,,,,,,,,
Atelectasis,50.0,0.595886,0.285577,0.029238,0.403491,0.681804,0.849257,0.959875
Cardiomegaly,50.0,0.468041,0.319891,0.021146,0.140346,0.492624,0.746844,0.980358
Effusion,50.0,0.776155,0.234438,0.023575,0.666477,0.867904,0.937628,0.991798
Pneumonia,50.0,0.316898,0.272006,0.009099,0.042073,0.215706,0.575757,0.748302
Pneumothorax,50.0,0.294373,0.188328,0.004224,0.126624,0.258001,0.505599,0.581488


### Findings & notes
**Model performance on the NIH subset is uneven and generally weaker than expected.** Only 31.2% of the 250 images were classified with a probability ≥ 0.7 for their true label, and this varies sharply by pathology:
- Effusion: mean 0.776
- Atelectasis: mean 0.596
- Cardiomegaly: mean 0.468
- Pneumonia: mean 0.317
- Pneumothorax: mean 0.294 

This is consistent with domain shift between the model's CheXpert training distribution and the NIH ChestX-ray14 test images used here, as expected in the Phase 1 research design.

**Inclusion rule for saliency generation (Phase 3 onward):** only images with a true-label probability ≥ 0.7 will be used. Because confident classification rates differ by pathology, this will leave unequal class sizes - Effusion and Atelectasis will retain the most images, while Pneumonia and Pneumothorax will retain far fewer. 

This imbalance is a direct consequence of the domain shift noted above and will be treated as a limitation, not corrected for artificially.